<a href="https://colab.research.google.com/github/Prajwaln08/SQLSmith/blob/main/notebooks/04_Qwen_7B_Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SQLSmith — 04 | Qwen2.5-Coder-7B Baseline

**Spider 1.0 first 50 development questions**

This notebook reproduces the **same 50 questions, full-schema prompt, deterministic generation, and official Spider test-suite settings** as Notebooks 01 and 03. No fine-tuning, schema linking, SQL repair or gold-assisted generation.

**Inputs:** `.../results/spider_1_5b_baseline_50.csv` and public Spider 1.0 data.  
**Persistent outputs:** `.../results/spider_7b_baseline_50.csv`, `.../results/spider_7b_baseline_50_checkpoint.jsonl`, and `.../results/spider_7b_official_subset_evaluation.json` (after official scoring).  
**Important:** first 50 examples cover only `concert_singer` and `pets_1`; results are **subset** comparisons, not full Spider development scores.


## 1 · Dependencies and Drive
Install only the packages needed here. Colab may request a runtime restart after package changes; if so, restart and rerun in order.

In [1]:
# Install dependencies before importing transformers or bitsandbytes.
# After an upgrade, restart the Colab session once if the runtime had imported these packages.
%pip install -q -U "transformers>=4.45,<5" "bitsandbytes>=0.46.1" accelerate pandas numpy huggingface_hub nltk sqlparse

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
numba 0.64.0 requires numpy<2.5,>=1.22, but you have numpy 2.5.5 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.


In [2]:
from google.colab import drive
from pathlib import Path
import os, gc, sys, json, hashlib, time, re, subprocess, zipfile, statistics
from datetime import datetime, timezone
import pandas as pd
import torch

drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/Prajwal/Projects/SQLSmith')
RESULTS_DIR = PROJECT_DIR / 'results'
EVAL_DIR = PROJECT_DIR / 'evaluation' / 'spider_50'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)
BASELINE_FILE = RESULTS_DIR / 'spider_1_5b_baseline_50.csv'
REPORT_1B_FILE = RESULTS_DIR / 'spider_1_5b_official_subset_evaluation.json'
CHECKPOINT_FILE = RESULTS_DIR / 'spider_7b_baseline_50_checkpoint.jsonl'
OUT_CSV = RESULTS_DIR / 'spider_7b_baseline_50.csv'
CONFIG_FILE = RESULTS_DIR / 'spider_7b_generation_config.json'
assert BASELINE_FILE.exists(), BASELINE_FILE
assert torch.cuda.is_available(), 'Enable a T4 GPU under Runtime > Change runtime type.'
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA memory total (GB):', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
print('Python:', sys.version.split()[0], 'PyTorch:', torch.__version__)
print('Transformers:', __import__('transformers').__version__)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Mounted at /content/drive
GPU: Tesla T4
CUDA memory total (GB): 14.56
Python: 3.13.16 PyTorch: 2.14.1+cu130
Transformers: 4.57.6


## 2 · Freeze the comparison set
The original 1.5B predictions define which first-50 Spider examples to evaluate. **Never use original gold SQL for generating 7B answers.**

In [3]:
df = pd.read_csv(BASELINE_FILE, keep_default_na=False)
required = {'index','db_id','question','gold_sql','predicted_sql','correct'}
assert required.issubset(df.columns), required - set(df.columns)
df['index'] = pd.to_numeric(df['index'], errors='raise').astype(int)
df = df.sort_values('index').reset_index(drop=True)
assert len(df) == 50 and df['index'].tolist() == list(range(1, 51))
assert df['db_id'].nunique() == 2, 'Unexpected database subset; investigate before comparing.'
SOURCE_HASH = sha256_file(BASELINE_FILE)
print('Original 1.5B file SHA256:', SOURCE_HASH)
print('DBs:', df['db_id'].value_counts().to_dict())
print('Subset size:', len(df))
display(df[['index','db_id','question']].head(5))

Original 1.5B file SHA256: 3d714667a99969fb16090359564da301d1b478eeba2bc1c376760d96fc953f3e
DBs: {'concert_singer': 45, 'pets_1': 5}
Subset size: 50


,index,db_id,question
0,1,concert_singer,How many singers do we have?
1,2,concert_singer,What is the total number of singers?
2,3,concert_singer,"Show name, country, age for all singers ordere..."
3,4,concert_singer,"What are the names, countries, and ages for ev..."
4,5,concert_singer,"What is the average, minimum, and maximum age ..."


## 3 · Load Spider metadata and build the **same full-schema prompt** as Notebook 01
Use the original `tables.json` and `dev.json`; the schema formatter deliberately matches the one used earlier. Before publishing comparisons, ensure your committed Notebook 01 used this exact formatter and prompt.

In [4]:
from huggingface_hub import hf_hub_download
SPIDER_DIR = Path('/content/sqlsmith_data/spider_data')
archive = hf_hub_download(repo_id='HAL-9001/spider-databases', repo_type='dataset', filename='spider_data.zip')
EXPECTED_HASH = '00636695dabed6b5f4b8328a16b13e069a2f16591d5efcce57660669c85b121b'
assert sha256_file(archive) == EXPECTED_HASH, 'Spider archive checksum changed'
if not (SPIDER_DIR / 'dev.json').exists() or not (SPIDER_DIR / 'tables.json').exists():
    root = Path('/content/sqlsmith_data').resolve()
    root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for m in z.infolist():
            assert (root / m.filename).resolve().is_relative_to(root), f'Unsafe ZIP path: {m.filename}'
        z.extractall(root)
with open(SPIDER_DIR / 'dev.json', encoding='utf-8') as f:
    dev = json.load(f)
with open(SPIDER_DIR / 'tables.json', encoding='utf-8') as f:
    metadata = json.load(f)
schemas = {item['db_id']:item for item in metadata}
for _, r in df.iterrows():
    original = dev[int(r['index'])-1]
    assert str(r['db_id']) == original['db_id']
    assert str(r['question']).strip() == original['question'].strip()
    assert ' '.join(str(r['gold_sql']).split()) == ' '.join(original['query'].split())
print('PASS: first-50 question, DB and gold SQL alignment verified')

def get_spider_schema(db_id):
    item = schemas[db_id]
    tables = item['table_names_original']
    columns = item['column_names_original']
    types = item['column_types']
    output = []
    for table_index, table_name in enumerate(tables):
        table_columns = []
        for col_index, (idx, col_name) in enumerate(columns):
            if idx == table_index:
                table_columns.append(f'  {col_name} {types[col_index]}')
        output.append(f'Table: {table_name}\n' + '\n'.join(table_columns))
    output.append('\nForeign keys:')
    for col_a, col_b in item['foreign_keys']:
        ta, ca = columns[col_a]
        tb, cb = columns[col_b]
        if ta >= 0 and tb >= 0:
            output.append(f'{tables[ta]}.{ca} = {tables[tb]}.{cb}')
    return '\n\n'.join(output)

print(get_spider_schema(df.iloc[0]['db_id'])[:1200])

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


spider_data.zip:   0%|          | 0.00/206M [00:00<?, ?B/s]

PASS: first-50 question, DB and gold SQL alignment verified
Table: stadium
  Stadium_ID number
  Location text
  Name text
  Capacity number
  Highest number
  Lowest number
  Average number

Table: singer
  Singer_ID number
  Name text
  Country text
  Song_Name text
  Song_release_year text
  Age number
  Is_male others

Table: concert
  concert_ID number
  concert_Name text
  Theme text
  Stadium_ID text
  Year text

Table: singer_in_concert
  concert_ID number
  Singer_ID text


Foreign keys:

concert.Stadium_ID = stadium.Stadium_ID

singer_in_concert.Singer_ID = singer.Singer_ID

singer_in_concert.concert_ID = concert.concert_ID


## 4 · Model and generation settings
**Match Notebook 01**: Qwen2.5-Coder instruct, 4-bit NF4 `float16`, `max_new_tokens=256`, `do_sample=False`, identical messages. We save settings before generating and refuse to resume a checkpoint created with conflicting settings.

In [5]:
MODEL_ID = 'Qwen/Qwen2.5-Coder-7B-Instruct'
MAX_NEW_TOKENS = 256
CONFIG = {
    'model': MODEL_ID,
    'quantization': 'bitsandbytes_4bit_nf4',
    'compute_dtype': 'float16',
    'prompt_revision': 'same_full_schema_spider_prompt_as_notebook01',
    'max_new_tokens': MAX_NEW_TOKENS,
    'do_sample': False,
    'subset': 'Spider dev first 50',
    'source_1b_sha256': SOURCE_HASH,
    'table_metadata_sha256': sha256_file(SPIDER_DIR / 'tables.json'),
}
if CONFIG_FILE.exists():
    existing = json.loads(CONFIG_FILE.read_text(encoding='utf-8'))
    assert existing == CONFIG, ('Existing 7B config differs. Do not mix experiments; archive the old artifacts first.')
else:
    assert not CHECKPOINT_FILE.exists(), 'Checkpoint exists without config; inspect it before resuming.'
    CONFIG_FILE.write_text(json.dumps(CONFIG, indent=2), encoding='utf-8')
print(json.dumps(CONFIG, indent=2))

{
  "model": "Qwen/Qwen2.5-Coder-7B-Instruct",
  "quantization": "bitsandbytes_4bit_nf4",
  "compute_dtype": "float16",
  "prompt_revision": "same_full_schema_spider_prompt_as_notebook01",
  "max_new_tokens": 256,
  "do_sample": false,
  "subset": "Spider dev first 50",
  "source_1b_sha256": "3d714667a99969fb16090359564da301d1b478eeba2bc1c376760d96fc953f3e",
  "table_metadata_sha256": "61bb20aa401f03164e2d7f3b16509b7b5f79cc9c943ca7bd159046df1159e2ed"
}


## Model loading — one 4-bit Qwen 7B instance

The original notebook contained duplicate model-loading cells and installed bitsandbytes a second time after imports. This corrected version installs all packages at the top, verifies GPU compatibility, and loads the model **once**. If you previously ran the old loading cell, use **Runtime → Restart session** before running from the top to free old weights.

In [6]:
import importlib.metadata as metadata
print('bitsandbytes:', metadata.version('bitsandbytes'))
print('transformers:', metadata.version('transformers'))
print('accelerate:', metadata.version('accelerate'))
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Select NVIDIA T4 GPU in Colab Runtime settings.'
print('GPU:', torch.cuda.get_device_name(0))

bitsandbytes: 0.50.2
transformers: 4.57.6
accelerate: 1.16.0
PyTorch: 2.14.1+cu130
CUDA: True
GPU: Tesla T4


In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Prevent accidentally loading a second copy in an existing session.
if 'model' in globals():
    del model
    gc.collect()
    torch.cuda.empty_cache()

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,  # matches 1.5B original 4-bit setup
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map='auto',
    dtype=torch.float16,
    low_cpu_mem_usage=True,
)
model.eval()
assert all(str(d) != 'cpu' for d in model.hf_device_map.values()), (
    'Detected CPU-offloaded layers; not comparable to T4-only inference.'
)
print('Model loaded:', MODEL_ID)
print('GPU memory allocated (GiB):', round(torch.cuda.memory_allocated() / 2**30, 2))
print('Device map:', model.hf_device_map)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

... (4 outputs left out)

model-00004-of-00004.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-Coder-7B-Instruct
GPU memory allocated (GiB): 5.46
Device map: {'': 0}


In [8]:
# Verify that the model is available and on the CUDA device before benchmarking.
assert 'model' in globals() and 'tokenizer' in globals()
assert model.training is False
assert torch.cuda.is_available()
print('PASS: one loaded 7B model is ready')
print('Allocated GPU memory (GiB):', round(torch.cuda.memory_allocated()/2**30, 2))

PASS: one loaded 7B model is ready
Allocated GPU memory (GiB): 5.46


## 5 · Generation and timing
Time **model generation only**, including autoregressive decoding, excluding Drive writes and SQL execution. We synchronize CUDA for accurate measurements. A resume continues from completed rows. Per-question checkpoint writes are durable, so Colab disconnects don't lose all work.

In [9]:
def generate_spider_sql(question, schema_text):
    messages = [
        {'role':'system','content':(
            'You are an expert text-to-SQL assistant. '
            'Generate exactly one valid SQLite SELECT query. '
            'Use only the tables and columns in the schema. '
            'Return SQL only, without markdown or explanation.'
        )},
        {'role':'user','content':f'Schema:\n{schema_text}\n\nQuestion: {question}'}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    batch = tokenizer(prompt, return_tensors='pt', add_special_tokens=False)
    input_ids = batch['input_ids'].to(model.device)
    attention_mask = batch['attention_mask'].to(model.device)
    torch.cuda.synchronize()
    started = time.perf_counter()
    with torch.inference_mode():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    new_tokens = outputs[0, input_ids.shape[-1]:]
    sql = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return sql, elapsed, int(new_tokens.numel()), int(input_ids.shape[-1])

# Print the prompt for comparison with committed Notebook 01, without invoking model inference.
probe = df.iloc[0]
print('First question:', probe['question'])
print('Schema characters:', len(get_spider_schema(probe['db_id'])))

First question: How many singers do we have?
Schema characters: 581


In [10]:
# Resume-safe append-only checkpoint. Failed attempts are saved as errors,
# but only successful predictions are skipped on resume.
completed = {}
if CHECKPOINT_FILE.exists():
    with open(CHECKPOINT_FILE, encoding='utf-8') as f:
        for lineno, line in enumerate(f, 1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                # A truncated final line after disconnect can be ignored.
                print('Warning: truncated checkpoint line', lineno)
                continue
            if record.get('status') == 'ok':
                ix = int(record['index'])
                assert 1 <= ix <= len(df), f'Unexpected checkpoint index {ix}'
                expected_row = df.iloc[ix - 1]
                assert record.get('db_id') == str(expected_row['db_id'])
                assert record.get('question') == str(expected_row['question'])
                completed[ix] = record
print(f'Resuming: {len(completed)}/50 completed predictions')

for _, row in df.iterrows():
    index = int(row['index'])
    if index in completed:
        print(f'[{index}/50] SKIP saved')
        continue
    db_id = str(row['db_id'])
    try:
        sql, latency, n_tokens, n_input = generate_spider_sql(
            str(row['question']), get_spider_schema(db_id)
        )
        record = {
            'index':index, 'db_id':db_id, 'question':str(row['question']),
            'predicted_sql':sql, 'latency_seconds':round(latency,5),
            'output_tokens':n_tokens, 'input_tokens':n_input,
            'status':'ok', 'error':'',
            'created_at_utc':datetime.now(timezone.utc).isoformat()
        }
    except Exception as exc:
        record = {
            'index':index, 'db_id':db_id, 'question':str(row['question']),
            'predicted_sql':'', 'latency_seconds':None,
            'output_tokens':None, 'input_tokens':None,
            'status':'error', 'error':repr(exc),
            'created_at_utc':datetime.now(timezone.utc).isoformat()
        }
    with open(CHECKPOINT_FILE, 'a', encoding='utf-8') as f:
        f.write(json.dumps(record, ensure_ascii=False) + '\n')
        f.flush()
        os.fsync(f.fileno())
    if record['status'] == 'ok':
        completed[index] = record
        print(f'[{index}/50] OK | {latency:.2f}s | {n_tokens} tokens')
    else:
        print(f'[{index}/50] ERROR:', record['error'])
        # Fail fast rather than silently corrupting a model benchmark.
        raise RuntimeError(f'Generation failed on question {index}. Retry this cell to resume.')

print('Complete:', len(completed), 'of', len(df))
assert len(completed) == 50

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Resuming: 0/50 completed predictions
[1/50] OK | 6.90s | 6 tokens
[2/50] OK | 0.59s | 6 tokens
[3/50] OK | 1.06s | 13 tokens
[4/50] OK | 0.89s | 13 tokens
[5/50] OK | 1.34s | 22 tokens
[6/50] OK | 1.45s | 22 tokens
[7/50] OK | 1.11s | 17 tokens
[8/50] OK | 1.33s | 22 tokens
[9/50] OK | 0.88s | 12 tokens
[10/50] OK | 0.86s | 12 tokens
[11/50] OK | 0.87s | 12 tokens
[12/50] OK | 0.87s | 12 tokens
[13/50] OK | 1.38s | 18 tokens
[14/50] OK | 1.38s | 18 tokens
[15/50] OK | 1.64s | 22 tokens
[16/50] OK | 1.41s | 23 tokens
[17/50] OK | 1.00s | 15 tokens
[18/50] OK | 1.06s | 16 tokens
[19/50] OK | 2.74s | 50 tokens
[20/50] OK | 2.81s | 50 tokens
[21/50] OK | 1.56s | 21 tokens
[22/50] OK | 1.59s | 21 tokens
[23/50] OK | 2.44s | 38 tokens
[24/50] OK | 2.20s | 38 tokens
[25/50] OK | 4.25s | 78 tokens
[26/50] OK | 3.41s | 57 tokens
[27/50] OK | 1.32s | 16 tokens
[28/50] OK | 1.32s | 16 tokens
[29/50] OK | 1.19s | 17 tokens
[30/50] OK | 1.20s | 17 tokens
[31/50] OK | 1.27s | 18 tokens
[32/50] OK | 

## 6 · Export frozen predictions and generation metrics
No gold SQL is shown to the model. The output CSV includes gold SQL strictly for downstream evaluation and alignment checks. SQL predictions are **not post-processed**.

In [11]:
records = [completed[i] for i in range(1, 51)]
result_df = df[['index','db_id','question','gold_sql']].merge(
    pd.DataFrame(records)[['index','predicted_sql','latency_seconds','output_tokens','input_tokens','status','error']],
    on='index', validate='one_to_one', how='left'
)
assert len(result_df) == 50 and result_df['predicted_sql'].notna().all()
assert not result_df['predicted_sql'].astype(str).str.strip().eq('').any(), (
    'Empty generation detected. Inspect the checkpoint; do not substitute gold SQL.'
)
result_df.to_csv(OUT_CSV, index=False)
latencies = result_df['latency_seconds'].astype(float)
result_df['tokens_per_sec'] = result_df['output_tokens'] / result_df['latency_seconds']
print('Saved:', OUT_CSV)
print('Predictions SHA256:', sha256_file(OUT_CSV))
print('Mean generation latency:', round(latencies.mean(),3),'s')
print('p95 generation latency:', round(latencies.quantile(0.95),3),'s')
print('Average generated tokens/s:', round(result_df['tokens_per_sec'].mean(),2))
print('Peak GPU memory allocated GB:', round(torch.cuda.max_memory_allocated()/2**30,2))
display(result_df[['index','db_id','predicted_sql','latency_seconds','output_tokens']].head())

Saved: /content/drive/MyDrive/Prajwal/Projects/SQLSmith/results/spider_7b_baseline_50.csv
Predictions SHA256: dcbbc6137a3e0bca1f6cc2baba15723fb89d180fd253ad3e50244514ca20b50d
Mean generation latency: 1.957 s
p95 generation latency: 4.088 s
Average generated tokens/s: 14.28
Peak GPU memory allocated GB: 6.58


,index,db_id,predicted_sql,latency_seconds,output_tokens
0,1,concert_singer,SELECT COUNT(*) FROM singer,6.89550,6
1,2,concert_singer,SELECT COUNT(*) FROM singer,0.59197,6
2,3,concert_singer,"SELECT Name, Country, Age FROM singer ORDER BY...",1.05903,13
3,4,concert_singer,"SELECT Name, Country, Age FROM singer ORDER BY...",0.88566,13
4,5,concert_singer,"SELECT AVG(Age), MIN(Age), MAX(Age) FROM singe...",1.33919,22


## 7 · Official Spider test-suite evaluation (optional if ZIP not yet available)
Use **the same** official augmented test-suite archive as Notebook 03. Because the public Google Drive link has download quota issues, this notebook deliberately **does not automatically call `gdown`**. Save a copy to:

`MyDrive/Prajwal/Projects/SQLSmith/data/testsuitedatabases.zip`

If you don't have it, prediction CSVs are already saved: you can stop here, restore the archive later, and run evaluation cells on a CPU-only runtime (without rerunning generation).

In [12]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
assert nltk.data.find('tokenizers/punkt_tab/english/')

EVALUATOR_DIR = Path('/content/test-suite-sql-eval')
if not (EVALUATOR_DIR / 'evaluation.py').is_file():
    subprocess.run(['git','clone','https://github.com/taoyds/test-suite-sql-eval.git',str(EVALUATOR_DIR)], check=True)
EVALUATOR_COMMIT = subprocess.check_output(
    ['git','-C',str(EVALUATOR_DIR),'rev-parse','HEAD'], text=True
).strip()
print('Evaluator commit:', EVALUATOR_COMMIT)

# Reuse persistent archive, never redownload from the rate-limited public Drive link.
DRIVE_ZIP = PROJECT_DIR / 'data' / 'testsuitedatabases.zip'
LOCAL_ZIP = Path('/content/testsuitedatabases.zip')
if DRIVE_ZIP.is_file():
    ARCHIVE = DRIVE_ZIP
elif LOCAL_ZIP.is_file():
    ARCHIVE = LOCAL_ZIP
else:
    raise FileNotFoundError(
        'Official ZIP not present. Save it as '+str(DRIVE_ZIP)+
        ' or upload to /content/testsuitedatabases.zip. Predictions are already saved.'
    )
assert zipfile.is_zipfile(ARCHIVE), f'Not a valid ZIP: {ARCHIVE}'
SUITE_EXTRACT = Path('/content/sqlsmith_test_suites')
SUITE_EXTRACT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ARCHIVE) as z:
    root = SUITE_EXTRACT.resolve()
    for m in z.infolist():
        assert (root / m.filename).resolve().is_relative_to(root), f'Unsafe ZIP path {m.filename}'
    z.extractall(root)
required_dbids = set(df['db_id'].astype(str))
possible = []
for anchor_id in required_dbids:
    for base_file in SUITE_EXTRACT.rglob(f'{anchor_id}.sqlite'):
        if base_file.parent.name != anchor_id:
            continue
        root = base_file.parent.parent
        if root not in possible and all((root/d/(d+'.sqlite')).is_file() for d in required_dbids):
            possible.append(root)
assert possible, 'No valid test-suite root found; verify archive contents.'
def count_suites(root):
    return sum(len(list((root/d).glob('*.sqlite'))) for d in required_dbids)
SUITE_DB_ROOT = max(possible, key=count_suites)
suite_counts = {d:len(list((SUITE_DB_ROOT/d).glob('*.sqlite'))) for d in sorted(required_dbids)}
assert max(suite_counts.values()) > 1, 'Augmented test-suite DB instances not found'
print('Test-suite DB root:', SUITE_DB_ROOT)
print('DB counts:', suite_counts)

Evaluator commit: e97acc546ecbee8fa27fa8dbf025ef61493a876c
Test-suite DB root: /content/sqlsmith_test_suites/database
DB counts: {'concert_singer': 32, 'pets_1': 27}


In [14]:

from pathlib import Path
import hashlib
import shutil

GOLD_FILE_1B = EVAL_DIR / "spider_50_gold.txt"
GOLD_FILE_7B = EVAL_DIR / "spider_50_gold_7b.txt"
PRED_FILE_7B = EVAL_DIR / "spider_50_predictions_7b.txt"

assert GOLD_FILE_1B.exists(), "Notebook 03 gold file missing"
assert len(result_df) == 50

def normalize_sql(sql):
    return " ".join(str(sql).split())

# Ensure predictions are ordered by original question index
ordered = result_df.sort_values("index").reset_index(drop=True)

assert ordered["index"].astype(int).tolist() == list(range(1, 51))

# Read the canonical gold file from Notebook 03
gold_lines = GOLD_FILE_1B.read_text(
    encoding="utf-8"
).splitlines()

assert len(gold_lines) == 50, "Expected 50 gold queries"

# Verify gold SQL and database IDs match across notebooks
for i, (line, row) in enumerate(
    zip(gold_lines, ordered.to_dict("records")),
    start=1
):
    gold_sql, db_id = line.rsplit("\t", 1)

    assert db_id == str(row["db_id"]), (
        f"Database mismatch at question {i}"
    )

    assert normalize_sql(gold_sql) == normalize_sql(
        row["gold_sql"]
    ), f"Gold SQL mismatch at question {i}"

print("PASS: All 50 gold queries and database IDs match.")

# Reuse exact Notebook 03 gold bytes
shutil.copyfile(GOLD_FILE_1B, GOLD_FILE_7B)

# Serialize 7B predictions in the same question order
predictions = [
    normalize_sql(sql)
    for sql in ordered["predicted_sql"]
]

assert all(predictions), "Found empty prediction"

PRED_FILE_7B.write_text(
    "\n".join(predictions) + "\n",
    encoding="utf-8"
)

def file_hash(path):
    return hashlib.sha256(
        path.read_bytes()
    ).hexdigest()

assert file_hash(GOLD_FILE_7B) == file_hash(GOLD_FILE_1B)

print("PASS: Gold file hashes match.")
print("Gold SHA256:", file_hash(GOLD_FILE_7B))
print("7B predictions:", len(predictions))
print("Section 7 ready.")


PASS: All 50 gold queries and database IDs match.
PASS: Gold file hashes match.
Gold SHA256: 2523871ac4fa312076a17448302d1778c254e5b072bb0e3cc509dd79e7eeb930
7B predictions: 50
Section 7 ready.


In [15]:
LOG_FILE_7B = EVAL_DIR / 'spider_50_7b_official_evaluation.log'
command = [
    sys.executable, str(EVALUATOR_DIR / 'evaluation.py'),
    '--gold', str(GOLD_FILE_7B), '--pred', str(PRED_FILE_7B),
    '--etype','exec', '--db', str(SUITE_DB_ROOT), '--table',str(SPIDER_DIR / 'tables.json')
]
print('Running:', ' '.join(command))
started = time.monotonic()
proc = subprocess.run(command, cwd=str(EVALUATOR_DIR), capture_output=True, text=True, timeout=1200)
eval_elapsed = time.monotonic()-started
LOG_FILE_7B.write_text('COMMAND: '+' '.join(command)+'\nSTDOUT:\n'+proc.stdout+'\nSTDERR:\n'+proc.stderr, encoding='utf-8')
print('Exit:',proc.returncode,'Elapsed:',round(eval_elapsed,1),'s')
print(proc.stdout[-7000:])
if proc.stderr:
    print('STDERR tail:',proc.stderr[-2500:])
assert proc.returncode == 0, f'Official evaluation failed: {LOG_FILE_7B}'
assert 'EXECUTION ACCURACY' in proc.stdout, 'Official evaluator did not show execution accuracy'
lines = proc.stdout.splitlines()
header_idx = next((i for i,line in enumerate(lines) if 'EXECUTION ACCURACY' in line),None)
score_line = next((line for line in lines[header_idx+1:] if re.match(r'^\s*execution\s+',line)),None)
assert score_line, 'Cannot find execution row in official evaluator output'
parts = score_line.strip().split()
assert len(parts) >= 6, f'Unexpected score row: {score_line}'
scores_by_difficulty = dict(zip(['easy','medium','hard','extra','all'],[float(x) for x in parts[1:6]]))
score_7b = scores_by_difficulty['all']
print('OFFICIAL 7B SUBSET TEST-SUITE ACCURACY:',f'{score_7b*100:.1f}%')

Running: /usr/bin/python3 /content/test-suite-sql-eval/evaluation.py --gold /content/drive/MyDrive/Prajwal/Projects/SQLSmith/evaluation/spider_50/spider_50_gold_7b.txt --pred /content/drive/MyDrive/Prajwal/Projects/SQLSmith/evaluation/spider_50/spider_50_predictions_7b.txt --etype exec --db /content/sqlsmith_test_suites/database --table /content/sqlsmith_data/spider_data/tables.json
Exit: 0 Elapsed: 5.3 s
                     easy                 medium               hard                 extra                all                 
count                6                    27                   13                   4                    50                  
=====================   EXECUTION ACCURACY     =====================
execution            1.000                0.630                0.769                0.250                0.680               

STDERR tail: /usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 

## 8 · Save official scoring report and compare
The comparison is valid only if the dataset, prompt, evaluation flags and test-suite databases are consistent with Notebook 03. 7B generation latency is single-request T4 latency, not a production p95 load-test measurement.

In [16]:
if REPORT_1B_FILE.exists():
    report_1b = json.loads(REPORT_1B_FILE.read_text(encoding='utf-8'))
    score_1b = float(report_1b['official_subset_accuracy'])
    assert report_1b['gold_file_sha256'] == sha256_file(GOLD_FILE_7B), '1.5B and 7B gold hashes differ'
    assert report_1b['test_suite_archive_sha256'] == sha256_file(ARCHIVE), 'Test-suite archive differs from Notebook 03'
else:
    score_1b = None
    print('1.5B report not found; skipping numerical comparison.')

report_7b = {
    'project':'SQLSmith', 'milestone':'A','notebook':'04_Qwen_7B_Baseline',
    'model':MODEL_ID, 'quantization':'4bit NF4',
    'dataset':'Spider 1.0 dev first 50', 'sample_size':50,
    'metric':'official test-suite execution accuracy (subset)',
    'official_subset_accuracy':score_7b,
    'scores_by_difficulty':scores_by_difficulty,
    'one_point_five_b_accuracy':score_1b,
    'accuracy_difference_percentage_points':None if score_1b is None else round((score_7b-score_1b)*100,3),
    'mean_generation_latency_seconds':float(latencies.mean()),
    'p95_generation_latency_seconds':float(latencies.quantile(0.95)),
    'mean_generated_tokens_per_second':float(result_df['tokens_per_sec'].mean()),
    'peak_gpu_memory_allocated_gib':round(torch.cuda.max_memory_allocated()/2**30,3),
    'plug_value':False,'keep_distinct':False,
    'evaluator_commit':EVALUATOR_COMMIT,
    'source_1b_sha256':SOURCE_HASH,
    'predictions_csv_sha256':sha256_file(OUT_CSV),
    'gold_file_sha256':sha256_file(GOLD_FILE_7B),
    'prediction_file_sha256':sha256_file(PRED_FILE_7B),
    'test_suite_archive_sha256':sha256_file(ARCHIVE),
    'test_suite_db_counts':suite_counts,
    'full_dev_evaluated':False,
    'generated_at_utc':datetime.now(timezone.utc).isoformat()
}
REPORT_7B_FILE = RESULTS_DIR / 'spider_7b_official_subset_evaluation.json'
REPORT_7B_FILE.write_text(json.dumps(report_7b,indent=2),encoding='utf-8')
print(json.dumps(report_7b,indent=2))
print('Saved report:', REPORT_7B_FILE)

{
  "project": "SQLSmith",
  "milestone": "A",
  "notebook": "04_Qwen_7B_Baseline",
  "model": "Qwen/Qwen2.5-Coder-7B-Instruct",
  "quantization": "4bit NF4",
  "dataset": "Spider 1.0 dev first 50",
  "sample_size": 50,
  "metric": "official test-suite execution accuracy (subset)",
  "official_subset_accuracy": 0.68,
  "scores_by_difficulty": {
    "easy": 1.0,
    "medium": 0.63,
    "hard": 0.769,
    "extra": 0.25,
    "all": 0.68
  },
  "one_point_five_b_accuracy": 0.42,
  "accuracy_difference_percentage_points": 26.0,
  "mean_generation_latency_seconds": 1.9569474000000002,
  "p95_generation_latency_seconds": 4.087536999999999,
  "mean_generated_tokens_per_second": 14.281956510974826,
  "peak_gpu_memory_allocated_gib": 6.584,
  "plug_value": false,
  "keep_distinct": false,
  "evaluator_commit": "e97acc546ecbee8fa27fa8dbf025ef61493a876c",
  "source_1b_sha256": "3d714667a99969fb16090359564da301d1b478eeba2bc1c376760d96fc953f3e",
  "predictions_csv_sha256": "dcbbc6137a3e0bca1f6cc2bab